# Phase A3.6 — GPU Validation on Google Colab (Tesla T4)

**This notebook automates *execution* of the existing avatar benchmark framework.**
It does **not** add models, redesign anything, or duplicate benchmark logic — every
step below shells out to code that already lives in the repository
(`avatar_engine.scripts.install_models`, `avatar_engine.scripts.run_benchmark`,
`avatar_engine.scripts.generate_scenario_audio`).

### What actually runs (honest, from the current codebase)
| Model | Status in the framework | On T4 |
|---|---|---|
| **SadTalker** | Full install spec (repo + checkpoints) + real adapter | **Runs and produces video** (needs the GPU-torch step + a real seed portrait — both automated below) |
| **LivePortrait** | Real adapter, but it is **video-driven** and its weights are not auto-fetched; the benchmark dataset is audio-driven | Attempted, then **skipped gracefully** (recorded with a reason) |
| **MuseTalk** | Registered as `PlannedAvatarAdapter` (no adapter implemented) | **Skipped gracefully** |
| **EchoMimic V3** | Registered as `PlannedAvatarAdapter` (no adapter implemented) | **Skipped gracefully** |

The pipeline attempts **SadTalker → LivePortrait → MuseTalk → EchoMimic V3** in order and
**continues past any model that cannot run**, exactly as the framework's registry intends.
Implementing the missing adapters would be new functionality and is explicitly out of scope.

### Reusability
To reuse on a future run, **edit only the `CONFIG` cell below.** Everything else is generic.

## 1. Configuration — *the only cell you edit between runs*

In [ ]:
# ============================ EDIT ME ============================
CONFIG = {
    # --- Repository (already on GitHub) -------------------------------------
    "REPO_URL": "https://github.com/nahatadhananjay33-svg/ai-creator-platform.git",
    "BRANCH":   "main",

    # --- Which models to attempt, in order ----------------------------------
    "MODELS": ["sadtalker", "liveportrait", "musetalk", "echomimic-v3"],

    # --- Benchmark scope ----------------------------------------------------
    "DEVICE":        "auto",   # "auto" -> cuda on the T4
    "CATEGORIES":    [],       # [] = all scenario categories
    "MAX_SCENARIOS": 2,        # 0 = all 10 scenarios; keep small for a quick smoke pass
    "REPETITIONS":   1,

    # --- Assets (the benchmark needs a real face + driving audio) -----------
    "ALLOW_PLACEHOLDER_ASSETS":     True,   # framework fills any gaps (faceless/tone stand-ins)
    "SEED_PORTRAITS_FROM_SADTALKER": True,  # copy SadTalker's own example faces so it can crop a face
    "GENERATE_REAL_AUDIO_KOKORO":    True,  # real speech via the Voice Engine (else sine tones)

    # --- GPU enablement for SadTalker ---------------------------------------
    # SadTalker's install spec pins CPU torch (basicsr compat). This reinstalls a
    # matching CUDA build *into SadTalker's isolated venv only* so the T4 is used.
    # It changes nothing in the repo. Set False to run SadTalker on CPU instead.
    "SADTALKER_GPU_TORCH": True,

    # --- Persistence / auth (optional) --------------------------------------
    "SAVE_TO_DRIVE": False,
    "DRIVE_DIR":     "/content/drive/MyDrive/ai_creator_platform_runs",
    "HF_TOKEN":      "",       # only if you hit a gated checkpoint
}
# ================================================================
for k, v in CONFIG.items():
    print(f"{k:28} = {v}")

## 2. Clone the repository

In [ ]:
import os, subprocess, sys
from pathlib import Path

assert "<your-username>" not in CONFIG["REPO_URL"], "Set CONFIG['REPO_URL'] to your GitHub repo first."

repo_name = CONFIG["REPO_URL"].rstrip("/").split("/")[-1].removesuffix(".git")
REPO_DIR = Path("/content") / repo_name

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", CONFIG["BRANCH"], "--depth", "1",
                    CONFIG["REPO_URL"], str(REPO_DIR)], check=True)
else:
    print("Repo already cloned; pulling latest.")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Working dir:", Path.cwd())

## 3. Verify CUDA & hardware (uses the framework's own probe)

In [ ]:
# Raw driver view
subprocess.run(["nvidia-smi"], check=False)

# The platform's own environment probe — the exact code the benchmark uses.
from foundation.model_manager.environment import probe_environment
env = probe_environment()
print("\n--- platform environment probe ---")
print("GPU(s):        ", [(g.name, g.vram_total_mb) for g in env.hardware.gpus])
print("Driver:        ", env.nvidia_driver.driver_version)
print("CUDA usable:   ", env.cuda_usable)
print("RAM (MB):      ", env.hardware.ram_total_mb)
print("Free disk (GB):", env.disk_free_gb)

CUDA_OK = env.cuda_usable
if not CUDA_OK:
    print("\n[warning] CUDA not usable — models will fall back to CPU where possible.")

## 4. Install tooling + platform core (fast, no model weights yet)

In [ ]:
# uv gives the installer a shared wheel cache (faster, resumable per-model venvs).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=False)
# Editable install so `foundation` / `avatar_engine` import in this Colab kernel.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Core installed. uv:", subprocess.run(["uv", "--version"], capture_output=True, text=True).stdout.strip())

## 5. (Optional) Mount Drive & Hugging Face login

In [ ]:
if CONFIG["SAVE_TO_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["DRIVE_DIR"]).mkdir(parents=True, exist_ok=True)
    # Persist HF cache across sessions to avoid re-downloading checkpoints.
    os.environ["HF_HOME"] = str(Path(CONFIG["DRIVE_DIR"]) / "hf_cache")
    print("Drive mounted; HF_HOME ->", os.environ["HF_HOME"])

if CONFIG["HF_TOKEN"]:
    from huggingface_hub import login
    login(CONFIG["HF_TOKEN"])
    print("Hugging Face: logged in.")

## 6. Install only the required model environments

Calls the existing installer CLI. For each model it creates an isolated venv, installs
**only that model's** dependency stack, clones its upstream repo, and prefetches **only its**
checkpoints (SadTalker's are prefetched automatically; other stacks download on first run).
The installer is **resumable** — re-running skips already-installed models.

In [ ]:
install_cmd = [sys.executable, "-m", "avatar_engine.scripts.install_models",
               "--models", *CONFIG["MODELS"]]
print(">", " ".join(install_cmd), "\n")
subprocess.run(install_cmd, check=False)   # per-model failures are captured in the report, not fatal

report = Path("avatar_engine/output/installs/installation_report.md")
if report.exists():
    print("\n================ INSTALLATION REPORT ================\n")
    print(report.read_text(encoding="utf-8"))

## 7. Enable the T4 for SadTalker *(venv-local, repo unchanged)*

SadTalker's pinned stack (torch 2.0.1 / torchvision 0.15.2) is CPU-only in the spec. This
reinstalls the **matching CUDA build** into SadTalker's venv so inference uses the T4. It is a
Colab-only enablement step — nothing in the repository changes. Skip by setting
`SADTALKER_GPU_TORCH = False`.

In [ ]:
sad_py = Path(".venvs/sadtalker/bin/python")
if CONFIG["SADTALKER_GPU_TORCH"] and CUDA_OK and sad_py.exists():
    cmd = ["uv", "pip", "install", "--python", str(sad_py),
           "torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2",
           "--index-url", "https://download.pytorch.org/whl/cu118"]
    print(">", " ".join(cmd), "\n")
    subprocess.run(cmd, check=False)
    chk = subprocess.run([str(sad_py), "-c", "import torch;print('sadtalker torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
                         capture_output=True, text=True)
    print(chk.stdout.strip() or chk.stderr.strip())
else:
    print("Skipped (disabled, no CUDA, or SadTalker venv missing). SadTalker will run on CPU.")

## 8. Seed real portraits *(so SadTalker can detect a face)*

Placeholder portraits are flat gray with no face, which SadTalker's cropper rejects. This copies
**SadTalker's own bundled example portraits** into the benchmark asset slots the scenarios expect.
These are existing example assets from the cloned repo — no new content is created.

In [ ]:
import shutil
if CONFIG["SEED_PORTRAITS_FROM_SADTALKER"]:
    examples = sorted(Path(".venvs/_repos/sadtalker/examples/source_image").glob("*.png"))
    assets_dir = Path("avatar_engine/datasets/data/assets"); assets_dir.mkdir(parents=True, exist_ok=True)
    targets = ["default_portrait.png", "portrait_three_quarter.png", "portrait_stylized.png"]
    if not examples:
        print("No SadTalker example portraits found (is SadTalker installed?). Falling back to placeholders.")
    else:
        for i, name in enumerate(targets):
            src = examples[i % len(examples)]
            shutil.copyfile(src, assets_dir / name)
            print(f"seeded {name:28} <- {src.name}")
else:
    print("Portrait seeding disabled; benchmark will use placeholder faces (SadTalker may fail to crop).")

## 9. Generate real driving audio with the Voice Engine *(optional but recommended)*

Runs the existing `generate_scenario_audio.py` inside the Kokoro voice venv, producing real
Kokoro speech for every scenario (the A3.5 methodology). If disabled or unavailable, the
benchmark falls back to the framework's sine-tone placeholders.

In [ ]:
if CONFIG["GENERATE_REAL_AUDIO_KOKORO"]:
    # Kokoro is a small CPU voice model; install its venv, then run the existing script in it.
    subprocess.run([sys.executable, "-m", "voice_engine.scripts.install_models", "--models", "kokoro"], check=False)
    kok_py = Path(".venvs/kokoro/bin/python")
    if kok_py.exists():
        r = subprocess.run([str(kok_py), "-m", "avatar_engine.scripts.generate_scenario_audio"],
                           capture_output=True, text=True)
        print(r.stdout[-2000:] or r.stderr[-2000:])
    else:
        print("Kokoro venv not available; using placeholder audio.")
else:
    print("Real-audio generation disabled; using placeholder audio.")

## 10. Verify model availability (framework's own `is_available()`)

In [ ]:
from avatar_engine.models import create_adapter
AVAILABILITY = {}
for mid in CONFIG["MODELS"]:
    try:
        adapter = create_adapter(mid, device=CONFIG["DEVICE"])
        ok = bool(adapter.is_available())
    except Exception as e:  # planned adapters / missing deps
        ok = False
    AVAILABILITY[mid] = ok
    print(f"{mid:16} {'AVAILABLE -> will run' if ok else 'not available -> will skip gracefully'}")

## 11. Execute the existing benchmark framework

Runs `avatar_engine.scripts.run_benchmark` **once per model, in order**, so a crash in one model
never stops the rest (`skip gracefully and continue`). Each call is the unmodified framework:
it builds cases from the scenario dataset, generates video, evaluates, and writes CSV/JSON/Markdown
reports under `avatar_engine/output/runs/<run_id>/`.

In [ ]:
def device_for(model_id):
    if model_id == "sadtalker":
        return "cuda" if (CONFIG["SADTALKER_GPU_TORCH"] and CUDA_OK) else "cpu"
    return CONFIG["DEVICE"]

RUN_RESULTS = {}   # model_id -> return code
for model_id in CONFIG["MODELS"]:
    dev = device_for(model_id)
    cmd = [sys.executable, "-m", "avatar_engine.scripts.run_benchmark",
           "--adapters", model_id, "--device", dev,
           "--max-scenarios", str(CONFIG["MAX_SCENARIOS"]),
           "--repetitions", str(CONFIG["REPETITIONS"])]
    if CONFIG["CATEGORIES"]:
        cmd += ["--categories", *CONFIG["CATEGORIES"]]
    print("\n" + "=" * 70)
    print(f"MODEL: {model_id}   (device={dev})")
    print("> " + " ".join(cmd))
    print("=" * 70)
    try:
        proc = subprocess.run(cmd, check=False)  # non-fatal; continue to next model
        RUN_RESULTS[model_id] = proc.returncode
    except Exception as e:
        RUN_RESULTS[model_id] = -1
        print(f"[skip] {model_id} raised: {e}")

print("\nReturn codes:", RUN_RESULTS)

## 12. Collect reports & generated videos

In [ ]:
runs_root = Path("avatar_engine/output/runs")
run_dirs = sorted([d for d in runs_root.glob("avatar-bench-*") if d.is_dir()],
                  key=lambda d: d.stat().st_mtime)

REPORTS, VIDEOS = [], []
for d in run_dirs:
    md_report = d / "report.md"
    md_report = md_report if md_report.exists() else next(iter(d.glob("*.md")), None)
    vids = sorted(list(d.glob("video/*.mp4")) + list(d.glob("video/*.avi")))
    REPORTS.append((d.name, md_report))
    VIDEOS.extend(vids)
    print(f"{d.name}: {len(vids)} video(s)" + (f"  report: {md_report.name}" if md_report else "  (no md report)"))

# Save generated media + reports to Drive if requested.
if CONFIG["SAVE_TO_DRIVE"] and VIDEOS:
    import shutil
    dest = Path(CONFIG["DRIVE_DIR"]) / "runs"; dest.mkdir(parents=True, exist_ok=True)
    for d in run_dirs:
        shutil.copytree(d, dest / d.name, dirs_exist_ok=True)
    print(f"\nCopied {len(run_dirs)} run dir(s) to {dest}")

### Latest markdown report(s) — framework-generated

In [ ]:
for name, rep in REPORTS[-len(CONFIG["MODELS"]):]:
    if rep and rep.exists():
        print("\n" + "#" * 70)
        print("#", name)
        print("#" * 70)
        print(rep.read_text(encoding="utf-8"))

## 13. Final summaries

In [ ]:
print("=" * 70); print("INSTALLATION SUMMARY"); print("=" * 70)
inst_json = Path("avatar_engine/output/installs/installation_report.json")
if inst_json.exists():
    import json
    data = json.loads(inst_json.read_text(encoding="utf-8"))
    for v in data.get("compatibility", []):
        print(f"  {v.get('model_id',''):16} compatible={v.get('compatible')} mode={v.get('mode')}")
    for r in data.get("installs", []):
        print(f"  {r.get('model_id',''):16} install={r.get('status')}  torch={r.get('torch_version')}")
else:
    print("  (installation_report.json not found)")

print("\n" + "=" * 70); print("MODEL AVAILABILITY / RUN OUTCOME"); print("=" * 70)
for m in CONFIG["MODELS"]:
    print(f"  {m:16} available={AVAILABILITY.get(m)}  benchmark_return={RUN_RESULTS.get(m)}")

print("\n" + "=" * 70); print("BENCHMARK SUMMARY (videos produced)"); print("=" * 70)
print(f"  total generated videos: {len(VIDEOS)}")
for v in VIDEOS:
    print("   ", v)

print("\n" + "=" * 70); print("GPU USAGE SUMMARY"); print("=" * 70)
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.used,memory.total,utilization.gpu",
                "--format=csv"], check=False)
try:
    import torch
    if torch.cuda.is_available():
        print(f"  torch peak allocated this kernel: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
except Exception:
    pass
print("  (per-scenario resource stats are in each run's CSV/JSON report.)")

print("\n" + "=" * 70); print("REPORT LOCATIONS"); print("=" * 70)
for name, rep in REPORTS[-len(CONFIG["MODELS"]):]:
    print(f"  {name}: {rep if rep else '(no md report)'}")
print("  installs: avatar_engine/output/installs/installation_report.md")
print("  runs dir: avatar_engine/output/runs/<run_id>/  (csv, json, markdown, video/)")
if CONFIG["SAVE_TO_DRIVE"]:
    print(f"  Drive copy: {CONFIG['DRIVE_DIR']}/runs/")

---
### Notes
- **Reusing this notebook:** change only the `CONFIG` cell (repo URL, model list, scenario cap, toggles).
- **Ephemeral storage:** Colab wipes `/content` between sessions. `.venvs/`, checkpoints, and
  `output/` are all git-ignored and do **not** persist — set `SAVE_TO_DRIVE=True` to keep reports/videos,
  and point `HF_HOME` at Drive to cache checkpoints.
- **Why only SadTalker produces video:** MuseTalk and EchoMimic V3 have no implemented adapter in the
  current codebase (they are `PlannedAvatarAdapter` stubs), and LivePortrait is video-driven while the
  dataset is audio-driven. Wiring those up is new functionality and out of scope for this phase.